# Import Statements

In [21]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/d-bert-train/transformers/default/1/d_bert_train/config.json
/kaggle/input/d-bert-train/transformers/default/1/d_bert_train/training_args.bin
/kaggle/input/d-bert-train/transformers/default/1/d_bert_train/tokenizer.json
/kaggle/input/d-bert-train/transformers/default/1/d_bert_train/tokenizer_config.json
/kaggle/input/d-bert-train/transformers/default/1/d_bert_train/model.safetensors
/kaggle/input/d-bert-train/transformers/default/1/d_bert_train/special_tokens_map.json
/kaggle/input/d-bert-train/transformers/default/1/d_bert_train/vocab.txt
/kaggle/input/2025-sep-dl-gen-ai-project/sample_submission.csv
/kaggle/input/2025-sep-dl-gen-ai-project/train.csv
/kaggle/input/2025-sep-dl-gen-ai-project/test.csv
/kaggle/input/distilbert-weights/distilbert-1/config.json
/kaggle/input/distilbert-weights/distilbert-1/tokenizer.json
/kaggle/input/distilbert-weights/distilbert-1/tokenizer_config.json
/kaggle/input/distilbert-weights/distilbert-1/model.safetensors
/kaggle/input/distilbert-

In [2]:
# !pip install contractions

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize
from wordcloud import WordCloud
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('punkt')

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

import torch
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from tqdm.auto import tqdm

2025-11-28 19:50:21.493141: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764359421.880707      38 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764359421.995211      38 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
df = pd.read_csv('/kaggle/input/2025-sep-dl-gen-ai-project/train.csv')
df.head()

,id,text,anger,fear,joy,sadness,surprise,emotions
0,0,the dentist that did the work apparently did a...,1,0,0,1,0,['anger' 'sadness']
1,1,i'm gonna absolutely ~~suck~~ be terrible duri...,0,1,0,1,0,['fear' 'sadness']
2,2,"bridge: so leave me drowning calling houston, ...",0,1,0,1,0,['fear' 'sadness']
3,3,after that mess i went to see my now ex-girlfr...,1,1,0,1,0,['anger' 'fear' 'sadness']
4,4,"as he stumbled i ran off, afraid it might some...",0,1,0,0,0,['fear']


In [5]:
emotion_cols = ['anger', 'fear', 'joy', 'sadness', 'surprise']
for col in emotion_cols:
    df[col] = df[col].astype(int)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [7]:
labels = emotion_cols
id2label = {idx: label for idx, label in enumerate(labels)}
label2id = {label: idx for idx, label in enumerate(labels)}

In [8]:
class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        labels = torch.tensor(self.labels[idx], dtype=torch.float32)

        # Tokenize the text
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': labels
        }

In [9]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
X_train = train_df['text'].tolist()
y_train = train_df[labels].values.tolist()
X_val = val_df['text'].tolist()
y_val = val_df[labels].values.tolist()

In [10]:
def compute_metrics(p):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    # Apply sigmoid and threshold
    sigmoid_preds = torch.sigmoid(torch.Tensor(preds))
    binary_preds = (sigmoid_preds > 0.5).int().numpy()
    
    true_labels = p.label_ids

    f1_macro = f1_score(y_true=true_labels, y_pred=binary_preds, average='macro', zero_division=0)
    
    return {
        'f1': f1_macro
    }

In [11]:
# MODEL_CHECKPOINT = 'bert-base-uncased'
# tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [12]:
# train_dataset = EmotionDataset(X_train, y_train, tokenizer)
# val_dataset = EmotionDataset(X_val, y_val, tokenizer)

In [13]:
# model = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_CHECKPOINT,
#     num_labels=len(labels),
#     id2label=id2label,
#     label2id=label2id,
#     problem_type="multi_label_classification"
# )
# model.to(device)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [16]:
os.environ["WANDB_DISABLED"] = "true"

In [17]:
# # Define training arguments
# training_args = TrainingArguments(
#     output_dir='./results',
#     num_train_epochs=3,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     learning_rate=2e-5,
#     weight_decay=0.01,
#     logging_dir='./logs',
#     logging_steps=100,
#     eval_strategy="epoch", # Evaluate at the end of each epoch
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     metric_for_best_model="f1",
#     report_to=None
# )

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [18]:
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset,
#     processing_class=tokenizer,
#     compute_metrics=compute_metrics
# )

# trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,F1
1,0.538500,0.376859,0.653101
2,0.343000,0.325362,0.735532
3,0.278600,0.311228,0.746907


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=513, training_loss=0.3717672253212734, metrics={'train_runtime': 258.1126, 'train_samples_per_second': 63.472, 'train_steps_per_second': 1.988, 'total_flos': 1077666131996928.0, 'train_loss': 0.3717672253212734, 'epoch': 3.0})

In [19]:
# model.save_pretrained("bert-1")
# tokenizer.save_pretrained("bert-1")

('bert-1/tokenizer_config.json',
 'bert-1/special_tokens_map.json',
 'bert-1/vocab.txt',
 'bert-1/added_tokens.json',
 'bert-1/tokenizer.json')

In [20]:
# !rm -rf logs
# !rm -rf results

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [22]:
MODEL_DIR = "/kaggle/input/bert-weights/bert-1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    problem_type="multi_label_classification"
)
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [23]:
inference_args = TrainingArguments(
    output_dir="/kaggle/working",
    per_device_eval_batch_size=16,
    dataloader_drop_last=False,
    report_to=None
)

trainer = Trainer(
    model=model,
    args=inference_args,
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [24]:
df_test = pd.read_csv('/kaggle/input/2025-sep-dl-gen-ai-project/test.csv')
test_texts = df_test['text'].tolist()

In [25]:
num_labels = len(emotion_cols)
dummy_labels = [[0] * num_labels for _ in range(len(test_texts))]

In [26]:
test_dataset = EmotionDataset(texts=test_texts, labels=dummy_labels, tokenizer=tokenizer)
raw_predictions = trainer.predict(test_dataset)

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


In [27]:
test_logits = raw_predictions.predictions
sigmoid_preds = torch.sigmoid(torch.Tensor(test_logits))
binary_preds = (sigmoid_preds > 0.5).int().numpy()
print(binary_preds[:5])

[[1 1 0 1 0]
 [0 0 0 0 0]
 [1 1 0 0 1]
 [0 1 0 0 0]
 [0 1 0 0 1]]


In [28]:
df_submission = pd.DataFrame(binary_preds, columns=emotion_cols)
df_submission[emotion_cols] = df_submission[emotion_cols].astype('int64')
df_submission['id'] = range(len(df_test))
df_submission = df_submission[['id', 'anger', 'fear', 'joy', 'sadness', 'surprise']]
df_submission.head()

,id,anger,fear,joy,sadness,surprise
0,0,1,1,0,1,0
1,1,0,0,0,0,0
2,2,1,1,0,0,1
3,3,0,1,0,0,0
4,4,0,1,0,0,1


In [29]:
df_submission.to_csv('submission.csv', index=False)